In [1]:
import numpy as np

def similaritetStringKernel(s, t, p):
    ngrama_s=set([s[i:i + p] for i in range(0, len(s) - p)])
    ngrama_t=set([t[i:i+p] for i in range(0, len(t) - p)])

    return  len(set(ngrama_s.intersection(ngrama_t)))

print(similaritetStringKernel("ananas copt","“banana verde",p=4))

2


In [3]:
test_data=np.load("data/test_data.npy")
train_data=np.load("data/train_data.npy")
train_label=np.load("data/train_labels.npy")

def classify(test):
    #calculam distantele imaginii test

    diferente=np.zeros(len(train_data))
    for i in range(len(train_data)):
        diferente[i]=similaritetStringKernel(train_data[i],test,p=8)

    #pastram cei mai apropiati k vecini
    labels=[]
    indici=np.argsort(diferente)[::-1]
    k=5
    for i in range(0,k):
        labels.append(train_label[indici[i]])

    givenLabel=max(labels)
    return givenLabel

f=open("solutie.txt","w")
for test in test_data:
    label=classify(test)
    print(label)
    f.write(str(label)+"\n")
f.close()

1
1
1
1
1
1
-1
1
1
1
1
1
1
1
-1
1
1
1
1
-1
1
1
1
1
-1
1
1
1
1
1
1
1
1
1
1
1
-1
1
-1
-1
-1
1
1
1
1
-1
-1
1
1
1
1
1
-1
1
1
1
1
-1
1
-1
1
1
1
-1
-1
1
1
1
1
1
-1
1
1
-1
1
-1
1
1
1
1
1
1
1
-1
1
1
1
1
-1
-1
1
1
-1
-1
1
1
1
1
1
1
1
1
-1
1
1
-1
-1
-1
1
1
1
1
1
1
-1
1
1
1
1
1
1
-1
1
1
1
1
-1
1
1
-1
1
1
1
-1
-1
1
1
1
-1
1
1
-1
1
1
1
1
1
-1
1
1
-1
1
1
-1
-1
-1
1
-1
-1
-1
1
1
-1
1
-1
1
1
1
-1
1
1
-1
1
1
1
-1
1
1
1
-1
1
-1
-1
1
1
-1
1
1
1
1
1
-1
1
1
1
1
1
-1
-1
1
-1
1
1
1
1
1
1
-1
-1
1
1
1
-1
1
1
-1
1
1
1
1
-1
1
1
1
1
1
1
1
1
1
1
1
1
1
-1
1
-1
-1
1
1
1
1
1
-1
1
-1
1
1
1
1
1
-1
-1
1
1
1
1
-1
1
-1
1
1
-1
-1
-1
-1
1
1
1
-1
1
1
1
-1
1
-1
-1
1
1
-1
1
1
-1
1
1
1
1
1
1
-1
1
1
1
1
-1
1
1
1
1
-1
1
1
1
1
1
1
-1
1
-1
1
1
-1
1
-1
1
1
-1
1
1
1
1
-1
1


In [4]:
def matriceKernel(x,z,p=4):
    K=np.zeros((len(x),len(z)))
    for i in range(len(x)):
        for j in range(len(z)):
            K[i][j]=similaritetStringKernel(x[i],z[j],p)
    return K

train_kernel=matriceKernel(train_data,train_data)
test_kernel=matriceKernel(train_data,test_data)
print(matriceKernel(train_data,train_data))
print(matriceKernel(train_data,test_data))

[[127.  16.   2. ...   1.  14.  11.]
 [ 16. 138.   7. ...   0.  20.   6.]
 [  2.   7.  49. ...   0.   9.  15.]
 ...
 [  1.   0.   0. ...  46.   0.   1.]
 [ 14.  20.   9. ...   0. 169.  19.]
 [ 11.   6.  15. ...   1.  19. 146.]]
[[ 0.  9.  5. ...  9.  3.  6.]
 [ 0.  6.  2. ...  3.  4.  4.]
 [ 0.  1.  1. ...  1.  9.  4.]
 ...
 [ 0.  0.  6. ...  8.  5.  0.]
 [ 0.  6. 14. ...  3. 36.  3.]
 [ 0.  2. 16. ...  7. 17. 11.]]


In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Împărțim datele de antrenare pentru validare (80-20)
train_data_split, val_data_split, train_label_split, val_label_split = train_test_split(
    train_data, train_label, test_size=0.2, random_state=42
)

# Calculam matricile kernel pentru split-ul de antrenare și validare
train_kernel_split = matriceKernel(train_data_split, train_data_split, p=8)
val_kernel_split = matriceKernel(val_data_split, train_data_split, p=8)

# Găsim parametrii optimi prin grid search
alphas = [0.001, 0.01, 0.1, 1, 10]
gammas = [0.001, 0.01, 0.1, 1]
best_score = -np.inf
best_params = {}

for alpha in alphas:
    for gamma in gammas:
        krr = KernelRidge(alpha=alpha, gamma=gamma, kernel="precomputed")
        krr.fit(train_kernel_split, train_label_split)
        score = krr.score(val_kernel_split, val_label_split)

        if score > best_score:
            best_score = score
            best_params = {'alpha': alpha, 'gamma': gamma}

print(f"Parametrii optimi: {best_params}")
print(f"Score validare: {best_score}")

# Antrenarea modelului final pe întreaga mulțime de antrenare
train_kernel_final = matriceKernel(train_data, train_data, p=8)
test_kernel_final = matriceKernel(test_data, train_data, p=8)

krr_model = KernelRidge(alpha=best_params['alpha'], gamma=best_params['gamma'], kernel="precomputed")
krr_model.fit(train_kernel_final, train_label)

# Predicții și generarea submisiilor
predictions = krr_model.predict(test_kernel_final)

# Submisia 1: Predicții brute
f1 = open("solutie_krr_brut.txt", "w")
for pred in predictions:
    f1.write(str(int(pred)) + "\n")
f1.close()

# Submisia 2: Predicții rotunjite
f2 = open("solutie_krr_rotunjit.txt", "w")
for pred in predictions:
    f2.write(str(round(pred)) + "\n")
f2.close()

# Submisia 3: Predicții cu threshold
f3 = open("solutie_krr_threshold.txt", "w")
for pred in predictions:
    f3.write(str(int(pred > 0.5)) + "\n")
f3.close()

print("Submisiile au fost generate: solutie_krr_brut.txt, solutie_krr_rotunjit.txt, solutie_krr_threshold.txt")